# MLP Sigmoid Experiment: Baseline vs Dynamic Sigmoid

This notebook compares MLP architectures using Sigmoid activations:

1. **Baseline (Sigmoid)**: Standard Sigmoid hidden layers + Softmax output
2. **Dynamic Sigmoid (Per-Neuron)**: Each neuron has learnable a,b parameters

**Training Approach**: 
- First train weights with baseline Sigmoid
- Then copy model, replace with Dynamic Sigmoid, and finetune activation parameters only

**Dynamic Sigmoid Function**: `sigmoid(b * (x - a))` where:
- `a` controls horizontal shift (threshold)
- `b` controls steepness/slope
- When a=0, b=1, reduces to standard sigmoid

In [1]:
import sys
import os

# Add src to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

from data_utils import DatasetConfig, DataManager
from mlp_trainer import MLPTrainer, ExperimentResult, MLPExperiment
from mlp import MLP, MLPConfig, create_baseline_sigmoid_mlp

print("Imports successful!")

Imports successful!


## Configuration

In [2]:
# Experiment Configuration
N_SEEDS = 20
SEEDS = [42, 123, 456, 789, 1001, 1111, 2222, 3333, 4444, 5555, 
         6666, 7777, 8888, 9999, 1010, 2020, 3030, 4040, 5050, 6060]
HIDDEN_DIMS = [256, 128]  # Two hidden layers
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 0.01
ACTIVATION_EPOCHS = 30

print("Configuration:")
print(f"  Number of seeds: {N_SEEDS}")
print(f"  Hidden layers: {HIDDEN_DIMS}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Activation training epochs: {ACTIVATION_EPOCHS}")

Configuration:
  Number of seeds: 20
  Hidden layers: [256, 128]
  Epochs: 30
  Batch size: 128
  Learning rate: 0.01
  Activation training epochs: 30


## Helper Classes and Functions

In [3]:
@dataclass
class AggregatedResult:
    """Aggregated results from multiple seeds."""
    model_name: str
    train_accs: List[float] = field(default_factory=list)
    test_accs: List[float] = field(default_factory=list)
    times: List[float] = field(default_factory=list)
    epochs: List[int] = field(default_factory=list)
    
    @property
    def train_mean(self) -> float:
        return np.mean(self.train_accs)
    
    @property
    def train_std(self) -> float:
        return np.std(self.train_accs)
    
    @property
    def test_mean(self) -> float:
        return np.mean(self.test_accs)
    
    @property
    def test_std(self) -> float:
        return np.std(self.test_accs)
    
    @property
    def time_mean(self) -> float:
        return np.mean(self.times)
    
    @property
    def epochs_mean(self) -> float:
        return np.mean(self.epochs)

In [4]:
def load_dataset(dataset_type: str, test_size: float = 0.2, random_state: int = 42):
    """Load and prepare dataset."""
    print(f"Loading {dataset_type} dataset...")
    config = DatasetConfig(
        dataset_type=dataset_type,
        test_size=test_size,
        random_state=random_state,
    )
    dm = DataManager(config)
    X_train, X_test, y_train, y_test = dm.generate_dataset()
    
    print(f"  Training samples: {len(X_train):,}")
    print(f"  Test samples: {len(X_test):,}")
    print(f"  Input features: {X_train.shape[1]}")
    print(f"  Classes: {len(np.unique(y_train))}")
    
    return X_train, X_test, y_train, y_test

In [5]:
def print_results(agg_results: Dict[str, AggregatedResult], dataset_name: str, n_seeds: int) -> None:
    """Print comparison of aggregated experiment results."""
    print("\n" + "=" * 100)
    print(f"EXPERIMENT RESULTS: {dataset_name.upper()} ({n_seeds} seeds)")
    print("=" * 100)
    
    headers = ["Model", "Train Acc", "Test Acc", "Time (s)", "Epochs"]
    row_format = "{:<45} {:>18} {:>18} {:>10} {:>8}"
    
    print(row_format.format(*headers))
    print("-" * 100)
    
    for name, r in agg_results.items():
        print(row_format.format(
            name[:45],
            f"{r.train_mean:.4f} ± {r.train_std:.4f}",
            f"{r.test_mean:.4f} ± {r.test_std:.4f}",
            f"{r.time_mean:.2f}",
            f"{r.epochs_mean:.1f}",
        ))
    
    print("-" * 100)
    
    # Highlight winner
    best_name = max(agg_results.keys(), key=lambda x: agg_results[x].test_mean)
    best = agg_results[best_name]
    baseline = agg_results.get("Baseline (Sigmoid)", list(agg_results.values())[0])
    improvement = best.test_mean - baseline.test_mean
    
    print(f"\n🏆 Best: {best_name}")
    print(f"   Test Accuracy: {best.test_mean:.4f} ± {best.test_std:.4f}")
    print(f"   Improvement over baseline: {improvement:+.4f} ({improvement*100:+.2f}%)")

In [6]:
def save_results(all_results: dict, filepath: str, n_seeds: int) -> None:
    """Save aggregated results to CSV file."""
    rows = []
    for dataset_name, agg_results in all_results.items():
        for model_name, r in agg_results.items():
            rows.append({
                "dataset": dataset_name,
                "model_name": model_name,
                "n_seeds": n_seeds,
                "train_acc_mean": r.train_mean,
                "train_acc_std": r.train_std,
                "test_acc_mean": r.test_mean,
                "test_acc_std": r.test_std,
                "time_mean_seconds": r.time_mean,
                "epochs_mean": r.epochs_mean,
                "train_accs": str(r.train_accs),
                "test_accs": str(r.test_accs),
                "timestamp": datetime.now().isoformat(),
            })
    
    df = pd.DataFrame(rows)
    df.to_csv(filepath, index=False)
    print(f"\nResults saved to: {filepath}")

## Sigmoid Experiment Functions

In [7]:
def run_sigmoid_baseline(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    hidden_dims: List[int],
    epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    verbose: bool = False,
) -> Tuple[MLP, ExperimentResult]:
    """
    Train baseline Sigmoid MLP.
    
    Returns both the trained model and the experiment result.
    """
    import time
    from sklearn.metrics import accuracy_score
    
    np.random.seed(seed)
    
    input_dim = X_train.shape[1]
    output_dim = len(np.unique(y_train))
    
    # Create baseline Sigmoid MLP
    model = create_baseline_sigmoid_mlp(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        output_dim=output_dim,
        learning_rate=learning_rate,
    )
    
    if verbose:
        print("\n" + "=" * 60)
        print("BASELINE MLP (Sigmoid)")
        print("=" * 60)
        print(model.summary())
    
    trainer = MLPTrainer(
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        verbose=verbose,
    )
    
    history = trainer.train(model, X_train, y_train, X_test, y_test)
    
    train_acc = trainer._compute_accuracy(model, X_train, y_train)
    test_acc = trainer._compute_accuracy(model, X_test, y_test)
    
    result = ExperimentResult(
        model_name="Baseline MLP (Sigmoid)",
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        training_time=history.training_time,
        epochs_trained=history.epochs_trained,
        model_params=model.num_params,
    )
    
    return model, result

In [8]:
def run_sigmoid_activation_finetuning(
    baseline_model: MLP,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    per_neuron: bool = True,
    verbose: bool = False,
) -> ExperimentResult:
    """
    Take a trained baseline Sigmoid model, copy it with Dynamic Sigmoid activations,
    and train only the activation parameters (weights frozen).
    """
    # Copy the baseline model with dynamic sigmoid activations
    dynamic_model = baseline_model.copy_with_dynamic_sigmoid_activations(
        per_neuron=per_neuron,
        activation_lr=learning_rate,
    )
    
    mode_str = "Per-Neuron" if per_neuron else "Per-Layer"
    
    if verbose:
        print("\n" + "=" * 60)
        print(f"SIGMOID ACTIVATION FINETUNING ({mode_str})")
        print("=" * 60)
        print("Starting from trained baseline Sigmoid weights...")
        print(dynamic_model.summary())
    
    trainer = MLPTrainer(
        epochs=activation_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        verbose=verbose,
    )
    
    # Check accuracy before activation training
    pre_train_acc = trainer._compute_accuracy(dynamic_model, X_test, y_test)
    if verbose:
        print(f"\nTest accuracy before activation training: {pre_train_acc:.4f}")
        print(f"\n--- Training activation parameters (weights frozen) ---")
    
    # Train only activation parameters
    activation_history = trainer.train_activation_params(
        dynamic_model, X_train, y_train,
        epochs=activation_epochs,
        X_val=X_test, y_val=y_test
    )
    
    train_acc = trainer._compute_accuracy(dynamic_model, X_train, y_train)
    test_acc = trainer._compute_accuracy(dynamic_model, X_test, y_test)
    
    if verbose:
        print(f"\nActivation training complete.")
        print(f"Test accuracy: {test_acc:.4f} (was {pre_train_acc:.4f}, improvement: {test_acc - pre_train_acc:+.4f})")
    
    # Collect activation info
    activation_info = []
    for layer in dynamic_model.layers:
        if hasattr(layer, 'activation') and layer.activation.is_learnable:
            activation_info.append(layer.activation.info)
    
    return ExperimentResult(
        model_name=f"Sigmoid Activation Finetuning ({mode_str})",
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        training_time=activation_history.training_time,
        epochs_trained=activation_history.epochs_trained,
        model_params=dynamic_model.num_params,
        activation_info="; ".join(activation_info),
    )

In [9]:
def run_single_seed_sigmoid(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    hidden_dims: List[int],
    epochs: int,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    verbose: bool = False,
) -> Dict[str, ExperimentResult]:
    """
    Run all Sigmoid experiment variants for a single seed.
    
    Approach: Train baseline Sigmoid once, then copy and finetune activations.
    """
    results = {}
    
    # 1. Train Baseline Sigmoid MLP
    if verbose:
        print("\n" + "🔷" * 30)
    
    baseline_model, baseline_result = run_sigmoid_baseline(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        hidden_dims=hidden_dims,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        seed=seed,
        verbose=verbose,
    )
    results["Baseline (Sigmoid)"] = baseline_result
    
    # 2. Copy baseline and finetune activations (Per-Neuron Dynamic Sigmoid)
    if verbose:
        print("\n" + "🔷" * 30)
    
    activation_result = run_sigmoid_activation_finetuning(
        baseline_model=baseline_model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        activation_epochs=activation_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        per_neuron=True,
        verbose=verbose,
    )
    results["Activation Finetuning (Per-Neuron)"] = activation_result
    
    return results

In [10]:
def run_sigmoid_experiment_suite(
    dataset_type: str,
    hidden_dims: List[int],
    epochs: int,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    seeds: List[int],
) -> Dict[str, AggregatedResult]:
    """Run Sigmoid experiments with multiple seeds and aggregate results."""
    
    # Load data once (use first seed for train/test split)
    X_train, X_test, y_train, y_test = load_dataset(dataset_type, random_state=seeds[0])
    
    # Initialize aggregated results
    model_names = [
        "Baseline (Sigmoid)",
        "Activation Finetuning (Per-Neuron)",
    ]
    agg_results = {name: AggregatedResult(name) for name in model_names}
    
    # Run experiments for each seed
    for i, seed in enumerate(seeds):
        print(f"\n  Seed {i+1}/{len(seeds)} (seed={seed})...")
        
        # Run all models with this seed
        seed_results = run_single_seed_sigmoid(
            X_train, X_test, y_train, y_test,
            hidden_dims=hidden_dims,
            epochs=epochs,
            activation_epochs=activation_epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
            seed=seed,
            verbose=False,
        )
        
        # Aggregate results
        for name, result in seed_results.items():
            agg_results[name].train_accs.append(result.train_accuracy)
            agg_results[name].test_accs.append(result.test_accuracy)
            agg_results[name].times.append(result.training_time)
            agg_results[name].epochs.append(result.epochs_trained)
        
        # Print progress
        baseline_acc = seed_results["Baseline (Sigmoid)"].test_accuracy
        best_acc = max(r.test_accuracy for r in seed_results.values())
        print(f"    Baseline: {baseline_acc:.4f}, Best: {best_acc:.4f}")
    
    return agg_results

## Run Experiments

In [11]:
print("=" * 100)
print("MLP SIGMOID EXPERIMENT: Baseline Sigmoid vs Dynamic Sigmoid Activation Finetuning")
print("Approach: Train baseline Sigmoid once, copy model, finetune activations on copy")
print("=" * 100)

all_results = {}

MLP SIGMOID EXPERIMENT: Baseline Sigmoid vs Dynamic Sigmoid Activation Finetuning
Approach: Train baseline Sigmoid once, copy model, finetune activations on copy


### CIFAR-10 Dataset

In [14]:
print("\n" + "🌟" * 40)
print("DATASET: CIFAR-10 (Color Images)")
print("🌟" * 40)

cifar_results = run_sigmoid_experiment_suite(
    dataset_type='cifar10',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['cifar10'] = cifar_results
print_results(cifar_results, "CIFAR-10", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: CIFAR-10 (Color Images)
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading cifar10 dataset...
CIFAR-10 loaded: 60000 samples, 3072 features
  Training samples: 48,000
  Test samples: 12,000
  Input features: 3072
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.4171, Best: 0.4171

  Seed 2/20 (seed=123)...
    Baseline: 0.4180, Best: 0.4180

  Seed 3/20 (seed=456)...
    Baseline: 0.4207, Best: 0.4212

  Seed 4/20 (seed=789)...
    Baseline: 0.4187, Best: 0.4197

  Seed 5/20 (seed=1001)...
    Baseline: 0.4173, Best: 0.4177

  Seed 6/20 (seed=1111)...
    Baseline: 0.4177, Best: 0.4186

  Seed 7/20 (seed=2222)...
    Baseline: 0.4212, Best: 0.4213

  Seed 8/20 (seed=3333)...
    Baseline: 0.4180, Best: 0.4182

  Seed 9/20 (seed=4444)...
    Baseline: 0.4168, Best: 0.4192

  Seed 10/20 (seed=5555)...
    Baseline: 0.4180, Best: 0.4206

  Seed 11/20 (seed=6666)...
    Baseline: 0.4189, Best: 0.4207

  Seed 12/20 (seed=7777)...
    B

### Fashion-MNIST Dataset (Optional)

In [13]:
# Uncomment to run on Fashion-MNIST
print("\n" + "🌟" * 40)
print("DATASET: FASHION-MNIST")
print("🌟" * 40)

fashion_results = run_sigmoid_experiment_suite(
    dataset_type='fashion_mnist',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['fashion_mnist'] = fashion_results
print_results(fashion_results, "Fashion-MNIST", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: FASHION-MNIST
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading fashion_mnist dataset...
  Training samples: 56,000
  Test samples: 14,000
  Input features: 784
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.8337, Best: 0.8340

  Seed 2/20 (seed=123)...
    Baseline: 0.8335, Best: 0.8354

  Seed 3/20 (seed=456)...
    Baseline: 0.8338, Best: 0.8338

  Seed 4/20 (seed=789)...
    Baseline: 0.8333, Best: 0.8341

  Seed 5/20 (seed=1001)...
    Baseline: 0.8350, Best: 0.8360

  Seed 6/20 (seed=1111)...
    Baseline: 0.8356, Best: 0.8357

  Seed 7/20 (seed=2222)...
    Baseline: 0.8336, Best: 0.8343

  Seed 8/20 (seed=3333)...
    Baseline: 0.8321, Best: 0.8329

  Seed 9/20 (seed=4444)...
    Baseline: 0.8343, Best: 0.8350

  Seed 10/20 (seed=5555)...
    Baseline: 0.8336, Best: 0.8346

  Seed 11/20 (seed=6666)...
    Baseline: 0.8362, Best: 0.8374

  Seed 12/20 (seed=7777)...
    Baseline: 0.8339, Best: 0.8346

  Seed 13/20 (seed=8

### MNIST Dataset (Optional)

In [12]:
# Uncomment to run on MNIST
print("\n" + "🌟" * 40)
print("DATASET: MNIST (Digits)")
print("🌟" * 40)

try:
    mnist_results = run_sigmoid_experiment_suite(
        dataset_type='mnist',
        hidden_dims=HIDDEN_DIMS,
        epochs=EPOCHS,
        activation_epochs=ACTIVATION_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        seeds=SEEDS,
    )
    all_results['mnist'] = mnist_results
    print_results(mnist_results, "MNIST", N_SEEDS)
except Exception as e:
    print(f"Skipping MNIST: {e}")


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: MNIST (Digits)
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading mnist dataset...
  Training samples: 56,000
  Test samples: 14,000
  Input features: 784
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.9127, Best: 0.9127

  Seed 2/20 (seed=123)...
    Baseline: 0.9128, Best: 0.9132

  Seed 3/20 (seed=456)...
    Baseline: 0.9131, Best: 0.9137

  Seed 4/20 (seed=789)...
    Baseline: 0.9144, Best: 0.9150

  Seed 5/20 (seed=1001)...
    Baseline: 0.9125, Best: 0.9131

  Seed 6/20 (seed=1111)...
    Baseline: 0.9132, Best: 0.9136

  Seed 7/20 (seed=2222)...
    Baseline: 0.9126, Best: 0.9126

  Seed 8/20 (seed=3333)...
    Baseline: 0.9121, Best: 0.9121

  Seed 9/20 (seed=4444)...
    Baseline: 0.9124, Best: 0.9125

  Seed 10/20 (seed=5555)...
    Baseline: 0.9129, Best: 0.9134

  Seed 11/20 (seed=6666)...
    Baseline: 0.9130, Best: 0.9130

  Seed 12/20 (seed=7777)...
    Baseline: 0.9110, Best: 0.9118

  Seed 13/20 (seed=8888)...

## Final Summary

In [15]:
print("\n" + "=" * 100)
print(f"FINAL SUMMARY (Mean ± Std over {N_SEEDS} seeds)")
print("=" * 100)

for dataset_name, agg_results in all_results.items():
    print(f"\n📊 {dataset_name.upper()}:")
    
    baseline = agg_results.get("Baseline (Sigmoid)")
    best_name = max(agg_results.keys(), key=lambda x: agg_results[x].test_mean)
    best = agg_results[best_name]
    improvement = best.test_mean - baseline.test_mean
    
    print(f"   Baseline (Sigmoid): {baseline.test_mean:.4f} ± {baseline.test_std:.4f}")
    print(f"   Best ({best_name}): {best.test_mean:.4f} ± {best.test_std:.4f}")
    print(f"   Improvement: {improvement:+.4f} ({improvement*100:+.2f}%)")


FINAL SUMMARY (Mean ± Std over 20 seeds)

📊 MNIST:
   Baseline (Sigmoid): 0.9128 ± 0.0009
   Best (Activation Finetuning (Per-Neuron)): 0.9131 ± 0.0010
   Improvement: +0.0003 (+0.03%)

📊 FASHION_MNIST:
   Baseline (Sigmoid): 0.8339 ± 0.0012
   Best (Activation Finetuning (Per-Neuron)): 0.8346 ± 0.0013
   Improvement: +0.0007 (+0.07%)

📊 CIFAR10:
   Baseline (Sigmoid): 0.4189 ± 0.0019
   Best (Activation Finetuning (Per-Neuron)): 0.4197 ± 0.0019
   Improvement: +0.0008 (+0.08%)


## Save Results

In [ ]:
output_path = "mlp_sigmoid_experiment_results.csv"
save_results(all_results, output_path, N_SEEDS)

# Display the saved results
df = pd.read_csv(output_path)
df